# 💳 Extração dos dados abertos — Cartão de Pagamento do Governo Federal (CPGF)

Este notebook foi preparado para **baixar, organizar, conferir e consolidar os arquivos mensais do Cartão de Pagamento do Governo Federal (CPGF)** disponibilizados pelo Portal da Transparência.

A lógica preserva o seguinte padrão de extração: o arquivo `.zip` é baixado **somente para o armazenamento temporário do Google Colab**, o `.csv` mensal é extraído diretamente para o Google Drive e, ao final da extração, o arquivo compactado é apagado.

O endereço de download segue o padrão:

`https://portaldatransparencia.gov.br/download-de-dados/cpgf/AAAAMM`

Em que `AAAA` representa o ano e `MM` representa o mês.

Nesta versão, o período completo considerado é de **janeiro de 2013 (`201301`) a julho de 2026 (`202607`)**, totalizando **163 competências mensais**.

Os CSVs mensais serão salvos diretamente em:

`/content/drive/MyDrive/Suprimentos de Fundos - CPGF`

e o arquivo consolidado será gravado em:

`/content/drive/MyDrive/Suprimentos de Fundos - CPGF/dados_consolidado`

> **Princípio de preservação:** os arquivos mensais são mantidos exatamente como publicados pelo Portal, com alteração apenas do nome do arquivo para o padrão `AAAAMM_CPGF.csv`. Nenhuma linha é eliminada, nenhuma duplicidade é removida e nenhuma coluna é transformada nesses arquivos brutos.

> **Proteção antirrobô:** o Portal da Transparência pode retornar uma página HTML, códigos HTTP de bloqueio temporário ou exigir validação pelo navegador. O notebook utiliza sessão HTTP, cabeçalhos usuais de navegador, pausas aleatórias e tentativas progressivas para reduzir requisições sucessivas. Ele **não tenta contornar CAPTCHA ou validações que exijam interação humana**; nesses casos, há uma alternativa segura de download manual.


## 1️⃣ Montar o Google Drive

Execute esta célula para permitir que o Colab salve os CSVs mensais e o arquivo consolidado diretamente no seu Google Drive.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2️⃣ Importar bibliotecas e configurar o projeto

Nesta etapa são definidas as pastas, o período completo da extração e os parâmetros de acesso ao Portal.

### 📁 Organização adotada

- pasta principal do projeto: contém diretamente os **163 CSVs mensais**;
- `dados_consolidado`: recebe o CSV único produzido ao final;
- `logs`: recebe arquivos de controle, validação e diagnóstico;
- pastas temporárias em `/content`: recebem os ZIPs e arquivos intermediários e são apagadas ao longo da execução.

Assim, os arquivos compactados **não são mantidos no Google Drive**.


In [2]:
import csv
import hashlib
import os
import random
import re
import shutil
import time
import zipfile
from pathlib import Path
from datetime import datetime
from typing import List, Optional, Dict, Tuple, Iterable

import pandas as pd
import requests


# ==========================
# 📁 Pastas do projeto
# ==========================

PROJECT_DIR = Path('/content/drive/MyDrive/Suprimentos de Fundos - CPGF')

# Os CSVs mensais ficam diretamente na pasta principal.
CSV_DIR = PROJECT_DIR

# Saídas auxiliares.
CONSOLIDADO_DIR = PROJECT_DIR / 'dados_consolidado'
LOG_DIR = PROJECT_DIR / 'logs'

# ZIPs nunca são mantidos no Drive.
TMP_ZIP_DIR = Path('/content/cpgf_tmp_zip')
TMP_EXTRACT_DIR = Path('/content/cpgf_tmp_extract')
MANUAL_ZIP_DIR = Path('/content/cpgf_zips_manuais')

for pasta in [
    PROJECT_DIR,
    CONSOLIDADO_DIR,
    LOG_DIR,
    TMP_ZIP_DIR,
    TMP_EXTRACT_DIR,
    MANUAL_ZIP_DIR,
]:
    pasta.mkdir(parents=True, exist_ok=True)


# ==========================
# 🧾 Arquivos de controle
# ==========================

LOG_FILE = LOG_DIR / 'download_cpgf.log'
CONTROLE_ARQUIVOS_CSV = LOG_DIR / 'controle_arquivos_cpgf.csv'
RELATORIO_ESQUEMAS_CSV = LOG_DIR / 'relatorio_esquemas_cpgf.csv'
RELATORIO_CONSOLIDACAO_CSV = LOG_DIR / 'relatorio_consolidacao_cpgf.csv'
VALIDACAO_FINAL_CSV = LOG_DIR / 'validacao_final_cpgf.csv'


# ==========================
# ⚙️ Configurações gerais
# ==========================

BASE_URL = 'https://portaldatransparencia.gov.br/download-de-dados/cpgf'

COMPETENCIA_INICIAL = '201301'
COMPETENCIA_FINAL = '202607'

# Pausa aleatória entre competências.
PAUSA_MIN_SEGUNDOS = 25
PAUSA_MAX_SEGUNDOS = 45

# Tentativas por competência.
MAX_TENTATIVAS = 3

# Espera adicional quando houver indício de bloqueio/HTML.
PAUSA_HTML_MIN_SEGUNDOS = 180
PAUSA_HTML_MAX_SEGUNDOS = 300

# Tempo máximo de uma requisição.
TIMEOUT_DOWNLOAD_SEGUNDOS = 1800

# Se True, um bloco anual para ao detectar HTML/bloqueio.
# A recuperação posterior pode ser usada para tentar novamente apenas os meses ausentes.
ABORTAR_BLOCO_SE_DETECTAR_BLOQUEIO = True

# Consolidação em blocos para reduzir uso de RAM.
CHUNKSIZE_CONSOLIDACAO = 250_000

# Por segurança, a consolidação só começa quando todos os 163 arquivos estiverem presentes.
EXIGIR_TODAS_COMPETENCIAS_PARA_CONSOLIDAR = True


def log(msg: str) -> None:
    """Registra uma mensagem na tela e no arquivo de log."""
    agora = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    linha = f'[{agora}] {msg}'
    print(linha)

    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(linha + '\\n')


log('✅ Configuração concluída.')
log(f'📁 Pasta principal: {PROJECT_DIR}')
log(f'📁 Pasta do consolidado: {CONSOLIDADO_DIR}')
log(f'📁 Pasta de logs: {LOG_DIR}')
log(f'📆 Período configurado: {COMPETENCIA_INICIAL} a {COMPETENCIA_FINAL}')


[2026-08-09 23:12:17] ✅ Configuração concluída.
[2026-08-09 23:12:17] 📁 Pasta principal: /content/drive/MyDrive/Suprimentos de Fundos - CPGF
[2026-08-09 23:12:17] 📁 Pasta do consolidado: /content/drive/MyDrive/Suprimentos de Fundos - CPGF/dados_consolidado
[2026-08-09 23:12:17] 📁 Pasta de logs: /content/drive/MyDrive/Suprimentos de Fundos - CPGF/logs
[2026-08-09 23:12:17] 📆 Período configurado: 201301 a 202607


## 3️⃣ Funções auxiliares de download e extração

As funções abaixo cuidam de sete tarefas principais:

1. gerar a lista de competências no formato `AAAAMM`;
2. criar uma sessão HTTP com cabeçalhos usuais de navegador;
3. baixar o ZIP mensal apenas para o armazenamento temporário do Colab;
4. identificar retorno HTML ou bloqueio temporário;
5. realizar novas tentativas com espera progressiva;
6. extrair somente o CSV e salvá-lo no Drive com nome padronizado;
7. apagar o ZIP imediatamente após a extração.

A função não substitui validações que dependam de JavaScript ou interação humana. Se isso ocorrer, utilize a seção de **download manual** mais adiante.


In [3]:
class PortalRetornouHTML(Exception):
    """O Portal retornou uma página HTML em vez do arquivo ZIP esperado."""
    pass


class PortalBloqueioTemporario(Exception):
    """O Portal sinalizou bloqueio temporário, como HTTP 403 ou 429."""
    pass


def validar_competencia(competencia: str) -> bool:
    """Confere se a competência está no formato AAAAMM."""
    return bool(re.fullmatch(r'20\d{2}(0[1-9]|1[0-2])', str(competencia)))


def proxima_competencia(competencia: str) -> str:
    """Retorna a competência seguinte no formato AAAAMM."""
    if not validar_competencia(competencia):
        raise ValueError(f'Competência inválida: {competencia}')

    ano = int(competencia[:4])
    mes = int(competencia[4:])

    if mes == 12:
        ano += 1
        mes = 1
    else:
        mes += 1

    return f'{ano}{mes:02d}'


def gerar_competencias(
    competencia_inicial: str,
    competencia_final: str
) -> List[str]:
    """Gera competências mensais entre duas datas, inclusive."""
    if not validar_competencia(competencia_inicial):
        raise ValueError(f'Competência inicial inválida: {competencia_inicial}')

    if not validar_competencia(competencia_final):
        raise ValueError(f'Competência final inválida: {competencia_final}')

    if competencia_inicial > competencia_final:
        raise ValueError('A competência inicial não pode ser maior que a competência final.')

    competencias = []
    atual = competencia_inicial

    while atual <= competencia_final:
        competencias.append(atual)
        atual = proxima_competencia(atual)

    return competencias


COMPETENCIAS_ESPERADAS = gerar_competencias(
    COMPETENCIA_INICIAL,
    COMPETENCIA_FINAL
)

print(f'📆 Total de competências esperadas: {len(COMPETENCIAS_ESPERADAS)}')


def caminho_csv_esperado(competencia: str) -> Path:
    """Retorna o caminho padronizado do CSV mensal no Drive."""
    return CSV_DIR / f'{competencia}_CPGF.csv'


def caminho_zip_tmp(competencia: str) -> Path:
    """Retorna o caminho temporário do ZIP no Colab."""
    return TMP_ZIP_DIR / f'{competencia}_CPGF.zip'


def arquivo_tem_assinatura_zip(caminho: Path) -> bool:
    """Verifica a assinatura mágica de um ZIP ('PK')."""
    caminho = Path(caminho)

    if not caminho.exists() or caminho.stat().st_size < 4:
        return False

    with open(caminho, 'rb') as f:
        return f.read(4).startswith(b'PK')


def ler_inicio_arquivo(caminho: Path, limite: int = 1500) -> str:
    """Lê apenas o início de um arquivo para diagnóstico."""
    try:
        with open(caminho, 'rb') as f:
            trecho = f.read(limite)

        return trecho.decode('utf-8', errors='replace')

    except Exception as e:
        return f'Não foi possível ler a prévia: {repr(e)}'


def criar_sessao_requests() -> requests.Session:
    """
    Cria uma sessão HTTP semelhante a uma navegação convencional.

    Os cabeçalhos servem para tornar a requisição mais completa e consistente.
    Eles não substituem validações por JavaScript, CAPTCHA ou interação humana.
    """
    sessao = requests.Session()

    sessao.headers.update({
        'User-Agent': (
            'Mozilla/5.0 (X11; Linux x86_64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/151.0 Safari/537.36'
        ),
        'Accept': (
            'application/zip,application/octet-stream,'
            'application/x-zip-compressed,text/html;q=0.8,*/*;q=0.7'
        ),
        'Accept-Language': 'pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7',
        'Connection': 'keep-alive',
        'Cache-Control': 'no-cache',
        'Pragma': 'no-cache',
        'Referer': BASE_URL,
    })

    return sessao


SESSAO = criar_sessao_requests()


def renovar_sessao() -> None:
    """Fecha a sessão atual e cria uma nova sessão HTTP."""
    global SESSAO

    try:
        SESSAO.close()
    except Exception:
        pass

    SESSAO = criar_sessao_requests()
    log('🔄 Sessão HTTP renovada.')


def pausa_aleatoria(
    minimo: int = PAUSA_MIN_SEGUNDOS,
    maximo: int = PAUSA_MAX_SEGUNDOS,
    motivo: str = 'antes da próxima requisição'
) -> int:
    """Aguarda um intervalo aleatório e retorna o tempo sorteado."""
    segundos = random.randint(minimo, maximo)
    log(f'😴 Aguardando {segundos}s {motivo}...')
    time.sleep(segundos)
    return segundos


def baixar_zip_temporario(
    competencia: str,
    timeout: int = TIMEOUT_DOWNLOAD_SEGUNDOS
) -> Path:
    """
    Baixa o ZIP mensal para o armazenamento temporário do Colab.

    Possíveis situações:
    - ZIP válido -> retorna o caminho;
    - HTML -> levanta PortalRetornouHTML;
    - HTTP 403/429 -> levanta PortalBloqueioTemporario;
    - outros erros HTTP -> requests levanta exceção.
    """
    if not validar_competencia(competencia):
        raise ValueError(f'Competência inválida: {competencia}')

    url = f'{BASE_URL}/{competencia}'
    destino = caminho_zip_tmp(competencia)

    if destino.exists():
        destino.unlink()

    log(f'⬇️ Iniciando download temporário de {competencia}: {url}')

    with SESSAO.get(
        url,
        stream=True,
        timeout=timeout,
        allow_redirects=True
    ) as resposta:

        content_type = resposta.headers.get('Content-Type', '')
        content_length = resposta.headers.get('Content-Length', 'não informado')

        log(
            f'🌐 HTTP {resposta.status_code} | '
            f'Content-Type: {content_type} | '
            f'Content-Length: {content_length}'
        )

        if resposta.status_code in (403, 429):
            previa = resposta.raw.read(1500, decode_content=True)
            texto = previa.decode('utf-8', errors='replace')

            print('\\n' + '-' * 80)
            print(texto[:1500])
            print('-' * 80 + '\\n')

            raise PortalBloqueioTemporario(
                f'O Portal retornou HTTP {resposta.status_code} para {competencia}.'
            )

        resposta.raise_for_status()

        with open(destino, 'wb') as f:
            for bloco in resposta.iter_content(chunk_size=1024 * 1024):
                if bloco:
                    f.write(bloco)

    tamanho_mb = destino.stat().st_size / (1024 ** 2)
    log(f'📦 Arquivo recebido: {tamanho_mb:.2f} MB')

    if arquivo_tem_assinatura_zip(destino):
        log(f'✅ ZIP válido recebido para {competencia}.')
        return destino

    previa = ler_inicio_arquivo(destino)
    previa_minuscula = previa.lower()

    if (
        '<html' in previa_minuscula
        or '<!doctype html' in previa_minuscula
        or 'javascript' in previa_minuscula
        or 'captcha' in previa_minuscula
        or 'robot' in previa_minuscula
        or 'verifica' in previa_minuscula
    ):
        log('⚠️ O Portal retornou HTML em vez de ZIP.')
        print('\\n' + '-' * 80)
        print(previa[:1500])
        print('-' * 80 + '\\n')

        try:
            destino.unlink()
        except Exception:
            pass

        raise PortalRetornouHTML(
            f'O Portal retornou HTML para {competencia}. '
            'Pode haver validação por navegador ou proteção antirrobô.'
        )

    print('\\n' + '-' * 80)
    print(previa[:1500])
    print('-' * 80 + '\\n')

    try:
        destino.unlink()
    except Exception:
        pass

    raise ValueError(
        f'O arquivo recebido para {competencia} não parece ser um ZIP válido.'
    )


def selecionar_csv_no_zip(
    z: zipfile.ZipFile,
    competencia: Optional[str] = None
) -> str:
    """
    Seleciona o CSV mais provável dentro do ZIP.

    Regras:
    1. se houver somente um CSV, usa esse arquivo;
    2. se houver vários, prioriza nome contendo a competência;
    3. persistindo empate, escolhe o maior CSV pelo tamanho descompactado.
    """
    membros_csv = [
        info for info in z.infolist()
        if not info.is_dir() and info.filename.lower().endswith('.csv')
    ]

    if not membros_csv:
        raise ValueError('Nenhum CSV foi encontrado dentro do ZIP.')

    if len(membros_csv) == 1:
        return membros_csv[0].filename

    candidatos = membros_csv

    if competencia:
        com_competencia = [
            info for info in membros_csv
            if competencia in Path(info.filename).name
        ]

        if com_competencia:
            candidatos = com_competencia

    escolhido = max(candidatos, key=lambda info: info.file_size)

    log(
        f'⚠️ O ZIP contém {len(membros_csv)} CSVs. '
        f'Foi selecionado: {Path(escolhido.filename).name}'
    )

    return escolhido.filename


def extrair_csv_do_zip(
    zip_path: Path,
    competencia: str,
    apagar_zip: bool = True
) -> Path:
    """
    Extrai o CSV do ZIP para o Google Drive.

    O conteúdo bruto não é modificado.
    Apenas o nome final é padronizado para AAAAMM_CPGF.csv.
    """
    zip_path = Path(zip_path)

    if not zip_path.exists():
        raise FileNotFoundError(f'ZIP não encontrado: {zip_path}')

    if not zipfile.is_zipfile(zip_path):
        raise ValueError(f'O arquivo não é um ZIP válido: {zip_path}')

    destino_csv = caminho_csv_esperado(competencia)

    if destino_csv.exists() and destino_csv.stat().st_size > 0:
        log(f'⏭️ CSV já existe no Drive: {destino_csv.name}')

        if apagar_zip and zip_path.exists():
            zip_path.unlink()
            log(f'🧹 ZIP temporário apagado: {zip_path.name}')

        return destino_csv

    if TMP_EXTRACT_DIR.exists():
        shutil.rmtree(TMP_EXTRACT_DIR)

    TMP_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as z:
        membro = selecionar_csv_no_zip(z, competencia=competencia)
        extraido_tmp = Path(z.extract(membro, path=TMP_EXTRACT_DIR))

    # Move o conteúdo sem reescrever o CSV.
    shutil.move(str(extraido_tmp), str(destino_csv))

    tamanho_mb = destino_csv.stat().st_size / (1024 ** 2)
    log(f'✅ CSV salvo: {destino_csv.name} ({tamanho_mb:.2f} MB)')

    try:
        shutil.rmtree(TMP_EXTRACT_DIR)
    except Exception:
        pass

    if apagar_zip and zip_path.exists():
        zip_path.unlink()
        log(f'🧹 ZIP temporário apagado: {zip_path.name}')

    return destino_csv


def processar_mes(
    competencia: str,
    max_tentativas: int = MAX_TENTATIVAS,
    espera_progressiva_base: int = 60
) -> bool:
    """
    Baixa, valida e extrai uma competência.

    Em erros temporários, renova a sessão e faz espera progressiva.
    """
    csv_path = caminho_csv_esperado(competencia)

    if csv_path.exists() and csv_path.stat().st_size > 0:
        log(f'⏭️ {competencia}: CSV já existe. Nada a fazer.')
        return True

    ultima_excecao = None

    for tentativa in range(1, max_tentativas + 1):
        log(f'🔁 {competencia} | tentativa {tentativa}/{max_tentativas}')

        try:
            zip_path = baixar_zip_temporario(competencia)
            extrair_csv_do_zip(
                zip_path,
                competencia=competencia,
                apagar_zip=True
            )

            if csv_path.exists() and csv_path.stat().st_size > 0:
                log(f'🎉 Competência concluída: {competencia}')
                return True

            raise RuntimeError(
                f'O processamento terminou, mas o CSV não foi localizado: {csv_path}'
            )

        except (PortalRetornouHTML, PortalBloqueioTemporario) as e:
            ultima_excecao = e
            log(f'🚧 {type(e).__name__}: {e}')

            renovar_sessao()

            if tentativa < max_tentativas:
                minimo = espera_progressiva_base * tentativa
                maximo = minimo + 60
                pausa_aleatoria(
                    minimo=minimo,
                    maximo=maximo,
                    motivo='antes de uma nova tentativa'
                )

        except (
            requests.RequestException,
            zipfile.BadZipFile,
            OSError,
            RuntimeError,
            ValueError
        ) as e:
            ultima_excecao = e
            log(f'❌ Erro em {competencia}: {repr(e)}')

            if tentativa < max_tentativas:
                minimo = 30 * tentativa
                maximo = minimo + 45
                pausa_aleatoria(
                    minimo=minimo,
                    maximo=maximo,
                    motivo='antes de uma nova tentativa'
                )

        finally:
            zip_tmp = caminho_zip_tmp(competencia)

            # Se algum arquivo inválido tiver permanecido, ele é removido.
            if zip_tmp.exists() and not arquivo_tem_assinatura_zip(zip_tmp):
                try:
                    zip_tmp.unlink()
                except Exception:
                    pass

    log(
        f'❌ Não foi possível concluir {competencia} '
        f'após {max_tentativas} tentativas.'
    )

    if isinstance(
        ultima_excecao,
        (PortalRetornouHTML, PortalBloqueioTemporario)
    ):
        raise ultima_excecao

    return False


def executar_bloco(
    competencia_inicial: str,
    competencia_final: str,
    abortar_em_bloqueio: bool = ABORTAR_BLOCO_SE_DETECTAR_BLOQUEIO
) -> Dict[str, List[str]]:
    """Executa uma faixa mensal com pausa aleatória entre as competências."""
    competencias = gerar_competencias(
        competencia_inicial,
        competencia_final
    )

    log(
        f'🚀 Iniciando bloco {competencia_inicial} a {competencia_final} '
        f'({len(competencias)} competência(s)).'
    )

    concluidas = []
    falhas = []
    bloqueadas = []

    for i, competencia in enumerate(competencias, start=1):
        log(f'📌 [{i}/{len(competencias)}] Competência {competencia}')

        try:
            sucesso = processar_mes(competencia)

            if sucesso:
                concluidas.append(competencia)
            else:
                falhas.append(competencia)

        except (PortalRetornouHTML, PortalBloqueioTemporario) as e:
            bloqueadas.append(competencia)
            log(f'🚧 Bloqueio/HTML detectado em {competencia}: {e}')

            if abortar_em_bloqueio:
                log(
                    '🛑 O bloco foi interrompido para evitar requisições '
                    'sucessivas durante a proteção do Portal.'
                )
                break

        except Exception as e:
            falhas.append(competencia)
            log(f'❌ Erro inesperado em {competencia}: {repr(e)}')

        if i < len(competencias):
            pausa_aleatoria()

    resumo = {
        'concluidas': concluidas,
        'falhas': falhas,
        'bloqueadas': bloqueadas,
    }

    log(
        f'📊 Resumo do bloco: '
        f'{len(concluidas)} concluída(s), '
        f'{len(falhas)} falha(s), '
        f'{len(bloqueadas)} bloqueio(s).'
    )

    return resumo


📆 Total de competências esperadas: 163


## 4️⃣ Funções para conferir estrutura, codificação e separador

A amostra de julho de 2026 contém 15 colunas. Elas são usadas apenas como **referência inicial de estrutura**.

A validação:

- lê somente o cabeçalho;
- não modifica o CSV;
- identifica a codificação;
- identifica o separador;
- compara o conjunto e a ordem das colunas;
- registra eventuais diferenças antes da consolidação.

Mesmo que algum arquivo histórico tenha uma estrutura diferente, a consolidação foi preparada para realizar uma **união flexível pelo nome das colunas**, preservando todas as colunas encontradas.


In [4]:
# Estrutura observada na amostra 202607_CPGF.csv.
COLUNAS_REFERENCIA_202607 = [
    'CÓDIGO ÓRGÃO SUPERIOR',
    'NOME ÓRGÃO SUPERIOR',
    'CÓDIGO ÓRGÃO',
    'NOME ÓRGÃO',
    'CÓDIGO UNIDADE GESTORA',
    'NOME UNIDADE GESTORA',
    'ANO EXTRATO',
    'MÊS EXTRATO',
    'CPF PORTADOR',
    'NOME PORTADOR',
    'CNPJ OU CPF FAVORECIDO',
    'NOME FAVORECIDO',
    'TRANSAÇÃO',
    'DATA TRANSAÇÃO',
    'VALOR TRANSAÇÃO',
]

CODIFICACOES_CANDIDATAS = [
    'utf-8-sig',
    'utf-8',
    'cp1252',
    'latin1',
]

SEPARADORES_CANDIDATOS = [';', ',', '\t']


def detectar_codificacao_csv(caminho_csv: Path) -> str:
    """Detecta uma codificação suficiente para leitura do arquivo."""
    caminho_csv = Path(caminho_csv)
    amostra = caminho_csv.read_bytes()[:100_000]

    for encoding in CODIFICACOES_CANDIDATAS:
        try:
            amostra.decode(encoding)
            return encoding
        except UnicodeDecodeError:
            continue

    return 'latin1'


def detectar_separador_csv(
    caminho_csv: Path,
    encoding: Optional[str] = None
) -> str:
    """Detecta o separador entre ; , e tabulação."""
    caminho_csv = Path(caminho_csv)

    if encoding is None:
        encoding = detectar_codificacao_csv(caminho_csv)

    with open(caminho_csv, 'r', encoding=encoding, errors='replace') as f:
        primeira_linha = f.readline()

    contagens = {
        sep: primeira_linha.count(sep)
        for sep in SEPARADORES_CANDIDATOS
    }

    return max(contagens, key=contagens.get)


def ler_cabecalho_csv(caminho_csv: Path) -> Tuple[List[str], str, str]:
    """Lê apenas o cabeçalho e retorna colunas, codificação e separador."""
    caminho_csv = Path(caminho_csv)

    encoding = detectar_codificacao_csv(caminho_csv)
    separador = detectar_separador_csv(caminho_csv, encoding=encoding)

    with open(
        caminho_csv,
        'r',
        encoding=encoding,
        newline=''
    ) as f:
        leitor = csv.reader(f, delimiter=separador)
        cabecalho = next(leitor)

    # Remove eventual BOM residual.
    if cabecalho:
        cabecalho[0] = cabecalho[0].lstrip('\ufeff')

    return cabecalho, encoding, separador


def validar_estrutura_csv(caminho_csv: Path) -> Dict[str, object]:
    """Valida um CSV contra a estrutura de referência de julho/2026."""
    caminho_csv = Path(caminho_csv)
    cabecalho, encoding, separador = ler_cabecalho_csv(caminho_csv)

    ausentes = [
        c for c in COLUNAS_REFERENCIA_202607
        if c not in cabecalho
    ]

    extras = [
        c for c in cabecalho
        if c not in COLUNAS_REFERENCIA_202607
    ]

    return {
        'arquivo': caminho_csv.name,
        'caminho': str(caminho_csv),
        'encoding': encoding,
        'separador': repr(separador),
        'numero_colunas': len(cabecalho),
        'cabecalho': cabecalho,
        'igual_referencia_conjunto': set(cabecalho) == set(COLUNAS_REFERENCIA_202607),
        'igual_referencia_ordem': cabecalho == COLUNAS_REFERENCIA_202607,
        'colunas_ausentes': ausentes,
        'colunas_extras': extras,
    }


## 5️⃣ Teste diagnóstico — julho de 2026

Antes de iniciar os 163 downloads, execute uma competência isolada.

Foi escolhido `202607` porque existe uma amostra desse período para comparação.

A célula:

1. tenta baixar o ZIP;
2. extrai apenas o CSV;
3. salva `202607_CPGF.csv` diretamente na pasta principal;
4. apaga o ZIP temporário;
5. confere o cabeçalho do CSV resultante.

Se o Portal retornar HTML ou bloquear temporariamente a requisição, a mensagem de diagnóstico será exibida.


In [5]:
# 🧪 Teste diagnóstico com julho de 2026

try:
    processar_mes('202607')
except (PortalRetornouHTML, PortalBloqueioTemporario) as e:
    print(f'🚧 Teste interrompido por proteção do Portal: {e}')

csv_teste = caminho_csv_esperado('202607')

if csv_teste.exists() and csv_teste.stat().st_size > 0:
    resultado_teste = validar_estrutura_csv(csv_teste)

    print('\n📋 Resultado da validação do CSV de teste:')
    for chave, valor in resultado_teste.items():
        if chave != 'cabecalho':
            print(f'- {chave}: {valor}')

    print('\n🧩 Cabeçalho encontrado:')
    print(resultado_teste['cabecalho'])
else:
    print('⚠️ O CSV 202607 ainda não está disponível no Drive.')


[2026-08-09 19:45:29] 🔁 202607 | tentativa 1/3
[2026-08-09 19:45:29] ⬇️ Iniciando download temporário de 202607: https://portaldatransparencia.gov.br/download-de-dados/cpgf/202607
[2026-08-09 19:45:30] 🌐 HTTP 200 | Content-Type: application/x-zip-compressed | Content-Length: 315081
[2026-08-09 19:45:30] 📦 Arquivo recebido: 0.30 MB
[2026-08-09 19:45:30] ✅ ZIP válido recebido para 202607.
[2026-08-09 19:45:30] ✅ CSV salvo: 202607_CPGF.csv (3.98 MB)
[2026-08-09 19:45:30] 🧹 ZIP temporário apagado: 202607_CPGF.zip
[2026-08-09 19:45:30] 🎉 Competência concluída: 202607

📋 Resultado da validação do CSV de teste:
- arquivo: 202607_CPGF.csv
- caminho: /content/drive/MyDrive/Suprimentos de Fundos - CPGF/202607_CPGF.csv
- encoding: cp1252
- separador: ';'
- numero_colunas: 15
- igual_referencia_conjunto: True
- igual_referencia_ordem: True
- colunas_ausentes: []
- colunas_extras: []

🧩 Cabeçalho encontrado:
['CÓDIGO ÓRGÃO SUPERIOR', 'NOME ÓRGÃO SUPERIOR', 'CÓDIGO ÓRGÃO', 'NOME ÓRGÃO', 'CÓDIGO UNID

## 6️⃣ Downloads anuais

A extração foi dividida em **uma célula por ano**.

Isso facilita:

- acompanhar o progresso;
- retomar o processamento após interrupção do Colab;
- identificar o ponto em que o Portal iniciou uma verificação antirrobô;
- evitar repetir competências que já foram baixadas.

Cada célula ignora automaticamente os CSVs que já estiverem salvos.


### 📦 2013 — janeiro a dezembro

Baixa as 12 competências de `201301` a `201312`.


In [6]:
executar_bloco('201301', '201312')


[2026-08-09 19:45:38] 🚀 Iniciando bloco 201301 a 201312 (12 competência(s)).
[2026-08-09 19:45:38] 📌 [1/12] Competência 201301
[2026-08-09 19:45:38] 🔁 201301 | tentativa 1/3
[2026-08-09 19:45:38] ⬇️ Iniciando download temporário de 201301: https://portaldatransparencia.gov.br/download-de-dados/cpgf/201301
[2026-08-09 19:45:39] 🌐 HTTP 200 | Content-Type: application/x-zip-compressed | Content-Length: 391846
[2026-08-09 19:45:40] 📦 Arquivo recebido: 0.37 MB
[2026-08-09 19:45:40] ✅ ZIP válido recebido para 201301.
[2026-08-09 19:45:40] ✅ CSV salvo: 201301_CPGF.csv (5.03 MB)
[2026-08-09 19:45:40] 🧹 ZIP temporário apagado: 201301_CPGF.zip
[2026-08-09 19:45:40] 🎉 Competência concluída: 201301
[2026-08-09 19:45:40] 😴 Aguardando 28s antes da próxima requisição...
[2026-08-09 19:46:08] 📌 [2/12] Competência 201302
[2026-08-09 19:46:08] 🔁 201302 | tentativa 1/3
[2026-08-09 19:46:08] ⬇️ Iniciando download temporário de 201302: https://portaldatransparencia.gov.br/download-de-dados/cpgf/201302
[202

{'concluidas': ['201301',
  '201302',
  '201303',
  '201304',
  '201305',
  '201306',
  '201307',
  '201308',
  '201309',
  '201310',
  '201311',
  '201312'],
 'falhas': [],
 'bloqueadas': []}

### 📦 2014 — janeiro a dezembro

Baixa as 12 competências de `201401` a `201412`.


In [7]:
executar_bloco('201401', '201412')


[2026-08-09 19:53:34] 🚀 Iniciando bloco 201401 a 201412 (12 competência(s)).
[2026-08-09 19:53:34] 📌 [1/12] Competência 201401
[2026-08-09 19:53:34] 🔁 201401 | tentativa 1/3
[2026-08-09 19:53:35] ⬇️ Iniciando download temporário de 201401: https://portaldatransparencia.gov.br/download-de-dados/cpgf/201401
[2026-08-09 19:53:35] 🌐 HTTP 200 | Content-Type: application/x-zip-compressed | Content-Length: 312694
[2026-08-09 19:53:36] 📦 Arquivo recebido: 0.30 MB
[2026-08-09 19:53:36] ✅ ZIP válido recebido para 201401.
[2026-08-09 19:53:36] ✅ CSV salvo: 201401_CPGF.csv (3.98 MB)
[2026-08-09 19:53:36] 🧹 ZIP temporário apagado: 201401_CPGF.zip
[2026-08-09 19:53:36] 🎉 Competência concluída: 201401
[2026-08-09 19:53:36] 😴 Aguardando 27s antes da próxima requisição...
[2026-08-09 19:54:03] 📌 [2/12] Competência 201402
[2026-08-09 19:54:03] 🔁 201402 | tentativa 1/3
[2026-08-09 19:54:03] ⬇️ Iniciando download temporário de 201402: https://portaldatransparencia.gov.br/download-de-dados/cpgf/201402
[202

{'concluidas': ['201401',
  '201402',
  '201403',
  '201404',
  '201405',
  '201406',
  '201407',
  '201408',
  '201409',
  '201410',
  '201411',
  '201412'],
 'falhas': [],
 'bloqueadas': []}

### 📦 2015 — janeiro a dezembro

Baixa as 12 competências de `201501` a `201512`.


In [8]:
executar_bloco('201501', '201512')


[2026-08-09 20:04:38] 🚀 Iniciando bloco 201501 a 201512 (12 competência(s)).
[2026-08-09 20:04:38] 📌 [1/12] Competência 201501
[2026-08-09 20:04:38] 🔁 201501 | tentativa 1/3
[2026-08-09 20:04:38] ⬇️ Iniciando download temporário de 201501: https://portaldatransparencia.gov.br/download-de-dados/cpgf/201501
[2026-08-09 20:04:39] 🌐 HTTP 200 | Content-Type: application/x-zip-compressed | Content-Length: 304912
[2026-08-09 20:04:40] 📦 Arquivo recebido: 0.29 MB
[2026-08-09 20:04:40] ✅ ZIP válido recebido para 201501.
[2026-08-09 20:04:40] ✅ CSV salvo: 201501_CPGF.csv (3.82 MB)
[2026-08-09 20:04:40] 🧹 ZIP temporário apagado: 201501_CPGF.zip
[2026-08-09 20:04:40] 🎉 Competência concluída: 201501
[2026-08-09 20:04:40] 😴 Aguardando 45s antes da próxima requisição...
[2026-08-09 20:05:25] 📌 [2/12] Competência 201502
[2026-08-09 20:05:25] 🔁 201502 | tentativa 1/3
[2026-08-09 20:05:25] ⬇️ Iniciando download temporário de 201502: https://portaldatransparencia.gov.br/download-de-dados/cpgf/201502
[202

{'concluidas': ['201501',
  '201502',
  '201503',
  '201504',
  '201505',
  '201506',
  '201507',
  '201508',
  '201509',
  '201510',
  '201511',
  '201512'],
 'falhas': [],
 'bloqueadas': []}

### 📦 2016 — janeiro a dezembro

Baixa as 12 competências de `201601` a `201612`.


In [9]:
executar_bloco('201601', '201612')


[2026-08-09 20:11:22] 🚀 Iniciando bloco 201601 a 201612 (12 competência(s)).
[2026-08-09 20:11:22] 📌 [1/12] Competência 201601
[2026-08-09 20:11:22] 🔁 201601 | tentativa 1/3
[2026-08-09 20:11:22] ⬇️ Iniciando download temporário de 201601: https://portaldatransparencia.gov.br/download-de-dados/cpgf/201601
[2026-08-09 20:11:23] 🌐 HTTP 200 | Content-Type: application/x-zip-compressed | Content-Length: 255850
[2026-08-09 20:11:23] 📦 Arquivo recebido: 0.24 MB
[2026-08-09 20:11:23] ✅ ZIP válido recebido para 201601.
[2026-08-09 20:11:23] ✅ CSV salvo: 201601_CPGF.csv (3.30 MB)
[2026-08-09 20:11:23] 🧹 ZIP temporário apagado: 201601_CPGF.zip
[2026-08-09 20:11:23] 🎉 Competência concluída: 201601
[2026-08-09 20:11:23] 😴 Aguardando 45s antes da próxima requisição...
[2026-08-09 20:12:08] 📌 [2/12] Competência 201602
[2026-08-09 20:12:08] 🔁 201602 | tentativa 1/3
[2026-08-09 20:12:08] ⬇️ Iniciando download temporário de 201602: https://portaldatransparencia.gov.br/download-de-dados/cpgf/201602
[202

{'concluidas': ['201601',
  '201602',
  '201603',
  '201604',
  '201605',
  '201606',
  '201607',
  '201608',
  '201609',
  '201610',
  '201611',
  '201612'],
 'falhas': [],
 'bloqueadas': []}

### 📦 2017 — janeiro a dezembro

Baixa as 12 competências de `201701` a `201712`.


In [10]:
executar_bloco('201701', '201712')


[2026-08-09 20:19:23] 🚀 Iniciando bloco 201701 a 201712 (12 competência(s)).
[2026-08-09 20:19:23] 📌 [1/12] Competência 201701
[2026-08-09 20:19:23] 🔁 201701 | tentativa 1/3
[2026-08-09 20:19:23] ⬇️ Iniciando download temporário de 201701: https://portaldatransparencia.gov.br/download-de-dados/cpgf/201701
[2026-08-09 20:19:24] 🌐 HTTP 200 | Content-Type: application/x-zip-compressed | Content-Length: 230477
[2026-08-09 20:19:25] 📦 Arquivo recebido: 0.22 MB
[2026-08-09 20:19:25] ✅ ZIP válido recebido para 201701.
[2026-08-09 20:19:25] ✅ CSV salvo: 201701_CPGF.csv (2.95 MB)
[2026-08-09 20:19:25] 🧹 ZIP temporário apagado: 201701_CPGF.zip
[2026-08-09 20:19:25] 🎉 Competência concluída: 201701
[2026-08-09 20:19:25] 😴 Aguardando 33s antes da próxima requisição...
[2026-08-09 20:19:58] 📌 [2/12] Competência 201702
[2026-08-09 20:19:58] 🔁 201702 | tentativa 1/3
[2026-08-09 20:19:58] ⬇️ Iniciando download temporário de 201702: https://portaldatransparencia.gov.br/download-de-dados/cpgf/201702
[202

{'concluidas': ['201701',
  '201702',
  '201703',
  '201704',
  '201705',
  '201706',
  '201707',
  '201708',
  '201709',
  '201710',
  '201711',
  '201712'],
 'falhas': [],
 'bloqueadas': []}

### 📦 2018 — janeiro a dezembro

Baixa as 12 competências de `201801` a `201812`.


In [11]:
executar_bloco('201801', '201812')


[2026-08-09 20:26:15] 🚀 Iniciando bloco 201801 a 201812 (12 competência(s)).
[2026-08-09 20:26:15] 📌 [1/12] Competência 201801
[2026-08-09 20:26:15] 🔁 201801 | tentativa 1/3
[2026-08-09 20:26:15] ⬇️ Iniciando download temporário de 201801: https://portaldatransparencia.gov.br/download-de-dados/cpgf/201801
[2026-08-09 20:26:15] 🌐 HTTP 200 | Content-Type: application/x-zip-compressed | Content-Length: 274964
[2026-08-09 20:26:16] 📦 Arquivo recebido: 0.26 MB
[2026-08-09 20:26:16] ✅ ZIP válido recebido para 201801.
[2026-08-09 20:26:16] ✅ CSV salvo: 201801_CPGF.csv (3.87 MB)
[2026-08-09 20:26:16] 🧹 ZIP temporário apagado: 201801_CPGF.zip
[2026-08-09 20:26:16] 🎉 Competência concluída: 201801
[2026-08-09 20:26:16] 😴 Aguardando 25s antes da próxima requisição...
[2026-08-09 20:26:41] 📌 [2/12] Competência 201802
[2026-08-09 20:26:41] 🔁 201802 | tentativa 1/3
[2026-08-09 20:26:41] ⬇️ Iniciando download temporário de 201802: https://portaldatransparencia.gov.br/download-de-dados/cpgf/201802
[202

{'concluidas': ['201801',
  '201802',
  '201803',
  '201804',
  '201805',
  '201806',
  '201807',
  '201808',
  '201809',
  '201810',
  '201811',
  '201812'],
 'falhas': [],
 'bloqueadas': []}

### 📦 2019 — janeiro a dezembro

Baixa as 12 competências de `201901` a `201912`.


In [12]:
executar_bloco('201901', '201912')


[2026-08-09 20:34:57] 🚀 Iniciando bloco 201901 a 201912 (12 competência(s)).
[2026-08-09 20:34:57] 📌 [1/12] Competência 201901
[2026-08-09 20:34:57] 🔁 201901 | tentativa 1/3
[2026-08-09 20:34:57] ⬇️ Iniciando download temporário de 201901: https://portaldatransparencia.gov.br/download-de-dados/cpgf/201901
[2026-08-09 20:34:57] 🌐 HTTP 200 | Content-Type: application/x-zip-compressed | Content-Length: 193477
[2026-08-09 20:34:58] 📦 Arquivo recebido: 0.18 MB
[2026-08-09 20:34:58] ✅ ZIP válido recebido para 201901.
[2026-08-09 20:34:58] ✅ CSV salvo: 201901_CPGF.csv (2.45 MB)
[2026-08-09 20:34:58] 🧹 ZIP temporário apagado: 201901_CPGF.zip
[2026-08-09 20:34:58] 🎉 Competência concluída: 201901
[2026-08-09 20:34:58] 😴 Aguardando 26s antes da próxima requisição...
[2026-08-09 20:35:24] 📌 [2/12] Competência 201902
[2026-08-09 20:35:24] 🔁 201902 | tentativa 1/3
[2026-08-09 20:35:24] ⬇️ Iniciando download temporário de 201902: https://portaldatransparencia.gov.br/download-de-dados/cpgf/201902
[202

{'concluidas': ['201901',
  '201902',
  '201903',
  '201904',
  '201905',
  '201906',
  '201907',
  '201908',
  '201909',
  '201910',
  '201911',
  '201912'],
 'falhas': [],
 'bloqueadas': []}

### 📦 2020 — janeiro a dezembro

Baixa as 12 competências de `202001` a `202012`.


In [13]:
executar_bloco('202001', '202012')


[2026-08-09 20:41:23] 🚀 Iniciando bloco 202001 a 202012 (12 competência(s)).
[2026-08-09 20:41:23] 📌 [1/12] Competência 202001
[2026-08-09 20:41:23] 🔁 202001 | tentativa 1/3
[2026-08-09 20:41:23] ⬇️ Iniciando download temporário de 202001: https://portaldatransparencia.gov.br/download-de-dados/cpgf/202001
[2026-08-09 20:41:24] 🌐 HTTP 200 | Content-Type: application/x-zip-compressed | Content-Length: 169298
[2026-08-09 20:41:24] 📦 Arquivo recebido: 0.16 MB
[2026-08-09 20:41:24] ✅ ZIP válido recebido para 202001.
[2026-08-09 20:41:24] ✅ CSV salvo: 202001_CPGF.csv (2.20 MB)
[2026-08-09 20:41:24] 🧹 ZIP temporário apagado: 202001_CPGF.zip
[2026-08-09 20:41:24] 🎉 Competência concluída: 202001
[2026-08-09 20:41:24] 😴 Aguardando 32s antes da próxima requisição...
[2026-08-09 20:41:56] 📌 [2/12] Competência 202002
[2026-08-09 20:41:56] 🔁 202002 | tentativa 1/3
[2026-08-09 20:41:56] ⬇️ Iniciando download temporário de 202002: https://portaldatransparencia.gov.br/download-de-dados/cpgf/202002
[202

{'concluidas': ['202001',
  '202002',
  '202003',
  '202004',
  '202005',
  '202006',
  '202007',
  '202008',
  '202009',
  '202010',
  '202011',
  '202012'],
 'falhas': [],
 'bloqueadas': []}

### 📦 2021 — janeiro a dezembro

Baixa as 12 competências de `202101` a `202112`.


In [14]:
executar_bloco('202101', '202112')


[2026-08-09 20:49:37] 🚀 Iniciando bloco 202101 a 202112 (12 competência(s)).
[2026-08-09 20:49:37] 📌 [1/12] Competência 202101
[2026-08-09 20:49:37] 🔁 202101 | tentativa 1/3
[2026-08-09 20:49:37] ⬇️ Iniciando download temporário de 202101: https://portaldatransparencia.gov.br/download-de-dados/cpgf/202101
[2026-08-09 20:49:37] 🌐 HTTP 200 | Content-Type: application/x-zip-compressed | Content-Length: 148489
[2026-08-09 20:49:38] 📦 Arquivo recebido: 0.14 MB
[2026-08-09 20:49:38] ✅ ZIP válido recebido para 202101.
[2026-08-09 20:49:38] ✅ CSV salvo: 202101_CPGF.csv (1.87 MB)
[2026-08-09 20:49:38] 🧹 ZIP temporário apagado: 202101_CPGF.zip
[2026-08-09 20:49:38] 🎉 Competência concluída: 202101
[2026-08-09 20:49:38] 😴 Aguardando 25s antes da próxima requisição...
[2026-08-09 20:50:03] 📌 [2/12] Competência 202102
[2026-08-09 20:50:03] 🔁 202102 | tentativa 1/3
[2026-08-09 20:50:03] ⬇️ Iniciando download temporário de 202102: https://portaldatransparencia.gov.br/download-de-dados/cpgf/202102
[202

{'concluidas': ['202101',
  '202102',
  '202103',
  '202104',
  '202105',
  '202106',
  '202107',
  '202108',
  '202109',
  '202110',
  '202111',
  '202112'],
 'falhas': [],
 'bloqueadas': []}

### 📦 2022 — janeiro a dezembro

Baixa as 12 competências de `202201` a `202212`.


In [15]:
executar_bloco('202201', '202212')


[2026-08-09 20:56:13] 🚀 Iniciando bloco 202201 a 202212 (12 competência(s)).
[2026-08-09 20:56:13] 📌 [1/12] Competência 202201
[2026-08-09 20:56:13] 🔁 202201 | tentativa 1/3
[2026-08-09 20:56:13] ⬇️ Iniciando download temporário de 202201: https://portaldatransparencia.gov.br/download-de-dados/cpgf/202201
[2026-08-09 20:56:14] 🌐 HTTP 200 | Content-Type: application/x-zip-compressed | Content-Length: 176990
[2026-08-09 20:56:14] 📦 Arquivo recebido: 0.17 MB
[2026-08-09 20:56:14] ✅ ZIP válido recebido para 202201.
[2026-08-09 20:56:14] ✅ CSV salvo: 202201_CPGF.csv (2.32 MB)
[2026-08-09 20:56:14] 🧹 ZIP temporário apagado: 202201_CPGF.zip
[2026-08-09 20:56:14] 🎉 Competência concluída: 202201
[2026-08-09 20:56:14] 😴 Aguardando 33s antes da próxima requisição...
[2026-08-09 20:56:47] 📌 [2/12] Competência 202202
[2026-08-09 20:56:47] 🔁 202202 | tentativa 1/3
[2026-08-09 20:56:47] ⬇️ Iniciando download temporário de 202202: https://portaldatransparencia.gov.br/download-de-dados/cpgf/202202
[202

{'concluidas': ['202201',
  '202202',
  '202203',
  '202204',
  '202205',
  '202206',
  '202207',
  '202208',
  '202209',
  '202210',
  '202211',
  '202212'],
 'falhas': [],
 'bloqueadas': []}

### 📦 2023 — janeiro a dezembro

Baixa as 12 competências de `202301` a `202312`.


In [16]:
executar_bloco('202301', '202312')


[2026-08-09 21:02:24] 🚀 Iniciando bloco 202301 a 202312 (12 competência(s)).
[2026-08-09 21:02:24] 📌 [1/12] Competência 202301
[2026-08-09 21:02:24] 🔁 202301 | tentativa 1/3
[2026-08-09 21:02:24] ⬇️ Iniciando download temporário de 202301: https://portaldatransparencia.gov.br/download-de-dados/cpgf/202301
[2026-08-09 21:02:25] 🌐 HTTP 200 | Content-Type: application/x-zip-compressed | Content-Length: 195798
[2026-08-09 21:02:25] 📦 Arquivo recebido: 0.19 MB
[2026-08-09 21:02:25] ✅ ZIP válido recebido para 202301.
[2026-08-09 21:02:25] ✅ CSV salvo: 202301_CPGF.csv (2.86 MB)
[2026-08-09 21:02:25] 🧹 ZIP temporário apagado: 202301_CPGF.zip
[2026-08-09 21:02:25] 🎉 Competência concluída: 202301
[2026-08-09 21:02:25] 😴 Aguardando 34s antes da próxima requisição...
[2026-08-09 21:02:59] 📌 [2/12] Competência 202302
[2026-08-09 21:02:59] 🔁 202302 | tentativa 1/3
[2026-08-09 21:02:59] ⬇️ Iniciando download temporário de 202302: https://portaldatransparencia.gov.br/download-de-dados/cpgf/202302
[202

{'concluidas': ['202301',
  '202302',
  '202303',
  '202304',
  '202305',
  '202306',
  '202307',
  '202308',
  '202309',
  '202310',
  '202311',
  '202312'],
 'falhas': [],
 'bloqueadas': []}

### 📦 2024 — janeiro a dezembro

Baixa as 12 competências de `202401` a `202412`.


In [17]:
executar_bloco('202401', '202412')


[2026-08-09 21:08:58] 🚀 Iniciando bloco 202401 a 202412 (12 competência(s)).
[2026-08-09 21:08:58] 📌 [1/12] Competência 202401
[2026-08-09 21:08:58] 🔁 202401 | tentativa 1/3
[2026-08-09 21:08:58] ⬇️ Iniciando download temporário de 202401: https://portaldatransparencia.gov.br/download-de-dados/cpgf/202401
[2026-08-09 21:08:59] 🌐 HTTP 200 | Content-Type: application/x-zip-compressed | Content-Length: 186543
[2026-08-09 21:08:59] 📦 Arquivo recebido: 0.18 MB
[2026-08-09 21:08:59] ✅ ZIP válido recebido para 202401.
[2026-08-09 21:08:59] ✅ CSV salvo: 202401_CPGF.csv (2.55 MB)
[2026-08-09 21:08:59] 🧹 ZIP temporário apagado: 202401_CPGF.zip
[2026-08-09 21:08:59] 🎉 Competência concluída: 202401
[2026-08-09 21:08:59] 😴 Aguardando 39s antes da próxima requisição...
[2026-08-09 21:09:38] 📌 [2/12] Competência 202402
[2026-08-09 21:09:38] 🔁 202402 | tentativa 1/3
[2026-08-09 21:09:38] ⬇️ Iniciando download temporário de 202402: https://portaldatransparencia.gov.br/download-de-dados/cpgf/202402
[202

{'concluidas': ['202401',
  '202402',
  '202403',
  '202404',
  '202405',
  '202406',
  '202407',
  '202408',
  '202409',
  '202410',
  '202411',
  '202412'],
 'falhas': [],
 'bloqueadas': []}

### 📦 2025 — janeiro a dezembro

Baixa as 12 competências de `202501` a `202512`.


In [18]:
executar_bloco('202501', '202512')


[2026-08-09 21:15:44] 🚀 Iniciando bloco 202501 a 202512 (12 competência(s)).
[2026-08-09 21:15:44] 📌 [1/12] Competência 202501
[2026-08-09 21:15:44] 🔁 202501 | tentativa 1/3
[2026-08-09 21:15:44] ⬇️ Iniciando download temporário de 202501: https://portaldatransparencia.gov.br/download-de-dados/cpgf/202501
[2026-08-09 21:15:45] 🌐 HTTP 200 | Content-Type: application/x-zip-compressed | Content-Length: 219642
[2026-08-09 21:15:45] 📦 Arquivo recebido: 0.21 MB
[2026-08-09 21:15:45] ✅ ZIP válido recebido para 202501.
[2026-08-09 21:15:45] ✅ CSV salvo: 202501_CPGF.csv (2.90 MB)
[2026-08-09 21:15:45] 🧹 ZIP temporário apagado: 202501_CPGF.zip
[2026-08-09 21:15:45] 🎉 Competência concluída: 202501
[2026-08-09 21:15:45] 😴 Aguardando 26s antes da próxima requisição...
[2026-08-09 21:16:11] 📌 [2/12] Competência 202502
[2026-08-09 21:16:11] 🔁 202502 | tentativa 1/3
[2026-08-09 21:16:11] ⬇️ Iniciando download temporário de 202502: https://portaldatransparencia.gov.br/download-de-dados/cpgf/202502
[202

{'concluidas': ['202501',
  '202502',
  '202503',
  '202504',
  '202505',
  '202506',
  '202507',
  '202508',
  '202509',
  '202510',
  '202511',
  '202512'],
 'falhas': [],
 'bloqueadas': []}

### 📦 2026 — janeiro a julho

Baixa as sete competências de `202601` a `202607`.


In [19]:
executar_bloco('202601', '202607')


[2026-08-09 21:21:55] 🚀 Iniciando bloco 202601 a 202607 (7 competência(s)).
[2026-08-09 21:21:55] 📌 [1/7] Competência 202601
[2026-08-09 21:21:55] 🔁 202601 | tentativa 1/3
[2026-08-09 21:21:55] ⬇️ Iniciando download temporário de 202601: https://portaldatransparencia.gov.br/download-de-dados/cpgf/202601
[2026-08-09 21:21:56] 🌐 HTTP 200 | Content-Type: application/x-zip-compressed | Content-Length: 178124
[2026-08-09 21:21:56] 📦 Arquivo recebido: 0.17 MB
[2026-08-09 21:21:56] ✅ ZIP válido recebido para 202601.
[2026-08-09 21:21:57] ✅ CSV salvo: 202601_CPGF.csv (2.67 MB)
[2026-08-09 21:21:57] 🧹 ZIP temporário apagado: 202601_CPGF.zip
[2026-08-09 21:21:57] 🎉 Competência concluída: 202601
[2026-08-09 21:21:57] 😴 Aguardando 44s antes da próxima requisição...
[2026-08-09 21:22:41] 📌 [2/7] Competência 202602
[2026-08-09 21:22:41] 🔁 202602 | tentativa 1/3
[2026-08-09 21:22:41] ⬇️ Iniciando download temporário de 202602: https://portaldatransparencia.gov.br/download-de-dados/cpgf/202602
[2026-0

{'concluidas': ['202601',
  '202602',
  '202603',
  '202604',
  '202605',
  '202606',
  '202607'],
 'falhas': [],
 'bloqueadas': []}

## 7️⃣ 🛡️ Recuperação das competências ausentes

Esta é a célula específica para situações em que o Portal da Transparência apresenta comportamento de proteção antirrobô.

Ela:

- identifica somente os meses que ainda não possuem CSV;
- renova a sessão HTTP antes da recuperação;
- usa novas tentativas;
- adota pausas aleatórias mais longas;
- diante de HTML/HTTP 403/429, aguarda de **3 a 5 minutos** antes de seguir;
- não insiste indefinidamente no mesmo mês.

> Se o Portal exigir CAPTCHA, JavaScript ou outra validação interativa, utilize a etapa de download manual. O notebook não tenta contornar esses mecanismos.


In [10]:
def listar_competencias_ausentes(
    competencia_inicial: str = COMPETENCIA_INICIAL,
    competencia_final: str = COMPETENCIA_FINAL
) -> List[str]:
    """Retorna as competências sem CSV válido no Google Drive."""
    ausentes = []

    for competencia in gerar_competencias(
        competencia_inicial,
        competencia_final
    ):
        csv_path = caminho_csv_esperado(competencia)

        if not csv_path.exists() or csv_path.stat().st_size == 0:
            ausentes.append(competencia)

    return ausentes


def executar_bloco_recuperacao(
    competencia_inicial: str = COMPETENCIA_INICIAL,
    competencia_final: str = COMPETENCIA_FINAL,
    limite_meses: Optional[int] = None
) -> None:
    """
    Tenta novamente apenas as competências ausentes.

    Ao detectar proteção do Portal:
    - renova a sessão;
    - espera de 3 a 5 minutos;
    - continua para a próxima competência ausente.
    """
    competencias = listar_competencias_ausentes(
        competencia_inicial,
        competencia_final
    )

    if limite_meses is not None:
        competencias = competencias[:limite_meses]

    if not competencias:
        log('✅ Não há competências remanescentes.')
        return

    renovar_sessao()

    log('🛡️ Iniciando recuperação das competências remanescentes.')
    log(f'📌 Total nesta execução: {len(competencias)}')

    concluidas = []
    falhas = []
    bloqueadas = []

    for i, competencia in enumerate(competencias, start=1):
        log(f'📌 [Recuperação {i}/{len(competencias)}] {competencia}')

        try:
            sucesso = processar_mes(
                competencia,
                max_tentativas=MAX_TENTATIVAS,
                espera_progressiva_base=90
            )

            if sucesso:
                concluidas.append(competencia)
            else:
                falhas.append(competencia)

        except (PortalRetornouHTML, PortalBloqueioTemporario) as e:
            bloqueadas.append(competencia)
            log(f'🚧 {competencia} ainda encontrou proteção do Portal: {e}')

            renovar_sessao()

            pausa_aleatoria(
                minimo=PAUSA_HTML_MIN_SEGUNDOS,
                maximo=PAUSA_HTML_MAX_SEGUNDOS,
                motivo='antes de seguir para outra competência ausente'
            )

            continue

        except Exception as e:
            falhas.append(competencia)
            log(f'❌ Erro inesperado em {competencia}: {repr(e)}')

        if i < len(competencias):
            pausa_aleatoria(
                minimo=45,
                maximo=90,
                motivo='antes da próxima competência remanescente'
            )

    ainda_ausentes = listar_competencias_ausentes(
        competencia_inicial,
        competencia_final
    )

    print('\\n' + '=' * 80)
    print('📊 RESUMO DA RECUPERAÇÃO')
    print('=' * 80)
    print(f'✅ Concluídas nesta execução: {len(concluidas)}')
    print(f'❌ Falhas comuns: {len(falhas)}')
    print(f'🚧 Bloqueadas/HTML: {len(bloqueadas)}')
    print(f'📂 Ainda ausentes: {len(ainda_ausentes)}')

    if ainda_ausentes:
        print('\\n📌 Competências ainda ausentes:')
        print(ainda_ausentes)


# ▶️ Execute quando houver competências ausentes.
executar_bloco_recuperacao(
    competencia_inicial=COMPETENCIA_INICIAL,
    competencia_final=COMPETENCIA_FINAL,
    limite_meses=None
)


[2026-08-09 23:15:40] ✅ Não há competências remanescentes.


## 8️⃣ Gerar links para download manual das competências ausentes

Se o Portal funcionar normalmente no navegador, mas bloquear as requisições do Colab, esta célula cria uma lista de links somente para os meses que continuam ausentes.

Os arquivos de apoio são gravados na pasta `logs`.

Os ZIPs baixados manualmente **não precisam ser salvos no Google Drive**. Na etapa seguinte, eles podem ser enviados diretamente ao armazenamento temporário do Colab e serão apagados após a extração.


In [11]:
competencias_para_download_manual = listar_competencias_ausentes(
    COMPETENCIA_INICIAL,
    COMPETENCIA_FINAL
)

html_path = LOG_DIR / 'links_download_manual_cpgf.html'
txt_path = LOG_DIR / 'links_download_manual_cpgf.txt'

if not competencias_para_download_manual:
    print('✅ Não há competências ausentes. Nenhum arquivo de links foi gerado.')

else:
    linhas_html = [
        '<!DOCTYPE html>',
        '<html lang="pt-BR">',
        '<head><meta charset="utf-8"><title>Links CPGF</title></head>',
        '<body>',
        '<h1>Links de download manual — CPGF</h1>',
        '<p>Use estes links somente quando o download automatizado pelo Colab não funcionar.</p>',
        '<ul>'
    ]

    linhas_txt = []

    for comp in competencias_para_download_manual:
        url = f'{BASE_URL}/{comp}'
        linhas_html.append(
            f'<li><a href="{url}" target="_blank">{comp}</a></li>'
        )
        linhas_txt.append(url)

    linhas_html.extend(['</ul>', '</body>', '</html>'])

    html_path.write_text(
        '\\n'.join(linhas_html),
        encoding='utf-8'
    )

    txt_path.write_text(
        '\\n'.join(linhas_txt),
        encoding='utf-8'
    )

    print(f'✅ HTML criado: {html_path}')
    print(f'✅ TXT criado: {txt_path}')
    print(f'📌 Total de links: {len(competencias_para_download_manual)}')
    print(competencias_para_download_manual)


✅ Não há competências ausentes. Nenhum arquivo de links foi gerado.


## 9️⃣ Alternativa segura — enviar ZIPs manualmente ao Colab

Use esta etapa apenas se o download automatizado continuar bloqueado.

### Procedimento

1. baixe pelo navegador os ZIPs correspondentes às competências ausentes;
2. execute a célula abaixo;
3. selecione um ou mais ZIPs no seu computador;
4. os arquivos serão enviados para `/content/cpgf_zips_manuais`;
5. o CSV será extraído diretamente para o Google Drive;
6. o ZIP temporário será apagado.

Assim, nenhum arquivo compactado permanece no Drive.


In [ ]:
from google.colab import files


def competencia_a_partir_do_nome(nome_arquivo: str) -> Optional[str]:
    """Tenta identificar AAAAMM no nome do arquivo."""
    m = re.search(r'(20\d{2}(0[1-9]|1[0-2]))', nome_arquivo)
    return m.group(1) if m else None


def enviar_e_extrair_zips_manuais() -> None:
    """Recebe ZIPs pelo navegador do Colab, extrai os CSVs e apaga os ZIPs."""
    MANUAL_ZIP_DIR.mkdir(parents=True, exist_ok=True)

    # Limpa resíduos de uma execução anterior.
    for arquivo in MANUAL_ZIP_DIR.iterdir():
        if arquivo.is_file():
            arquivo.unlink()

    print('📤 Selecione um ou mais ZIPs baixados manualmente do Portal.')
    uploaded = files.upload()

    if not uploaded:
        print('⚠️ Nenhum arquivo foi enviado.')
        return

    zips_recebidos = []

    for nome, conteudo in uploaded.items():
        destino = MANUAL_ZIP_DIR / Path(nome).name
        destino.write_bytes(conteudo)
        zips_recebidos.append(destino)

    print(f'📦 Arquivos recebidos: {len(zips_recebidos)}')

    for zip_path in zips_recebidos:
        competencia = competencia_a_partir_do_nome(zip_path.name)

        if not competencia:
            log(
                f'⚠️ Não foi possível identificar AAAAMM no nome: '
                f'{zip_path.name}'
            )
            continue

        try:
            extrair_csv_do_zip(
                zip_path,
                competencia=competencia,
                apagar_zip=True
            )

        except Exception as e:
            log(f'❌ Falha na extração manual de {zip_path.name}: {repr(e)}')

    # Limpeza final.
    for arquivo in MANUAL_ZIP_DIR.iterdir():
        if arquivo.is_file():
            try:
                arquivo.unlink()
            except Exception:
                pass

    print('🧹 Limpeza dos ZIPs temporários concluída.')


# ▶️ Execute somente se precisar do procedimento manual.
# enviar_e_extrair_zips_manuais()


## 🔟 Conferência dos 163 arquivos mensais

Esta etapa verifica:

- quantas competências deveriam existir;
- quantas estão presentes;
- quais ainda estão ausentes;
- tamanho individual dos arquivos;
- volume total armazenado.

Ela gera o arquivo:

`logs/controle_arquivos_cpgf.csv`

A contagem detalhada de registros de cada CSV será produzida durante a consolidação, evitando uma leitura completa duplicada dos 163 arquivos.


In [8]:
def gerar_controle_arquivos(
    competencia_inicial: str = COMPETENCIA_INICIAL,
    competencia_final: str = COMPETENCIA_FINAL
) -> List[Dict[str, object]]:
    """Gera o controle das competências mensais presentes e ausentes."""
    competencias = gerar_competencias(
        competencia_inicial,
        competencia_final
    )

    controle = []

    for comp in competencias:
        csv_path = caminho_csv_esperado(comp)
        existe = csv_path.exists() and csv_path.stat().st_size > 0
        tamanho_bytes = csv_path.stat().st_size if existe else 0

        controle.append({
            'competencia': comp,
            'arquivo_esperado': csv_path.name,
            'status': 'presente' if existe else 'ausente',
            'tamanho_bytes': tamanho_bytes,
            'tamanho_mb': round(tamanho_bytes / (1024 ** 2), 2),
            'caminho': str(csv_path),
        })

    with open(
        CONTROLE_ARQUIVOS_CSV,
        'w',
        encoding='utf-8-sig',
        newline=''
    ) as f:

        campos = [
            'competencia',
            'arquivo_esperado',
            'status',
            'tamanho_bytes',
            'tamanho_mb',
            'caminho',
        ]

        escritor = csv.DictWriter(
            f,
            fieldnames=campos,
            delimiter=';'
        )

        escritor.writeheader()
        escritor.writerows(controle)

    return controle


controle = gerar_controle_arquivos()

presentes = [
    r for r in controle
    if r['status'] == 'presente'
]

ausentes = [
    r['competencia'] for r in controle
    if r['status'] == 'ausente'
]

tamanho_total_bytes = sum(
    r['tamanho_bytes'] for r in controle
)

tamanho_total_gb = tamanho_total_bytes / (1024 ** 3)

print(f'✅ CSVs presentes: {len(presentes)} de {len(controle)}')
print(f'❌ CSVs ausentes: {len(ausentes)}')
print(f'💾 Volume aproximado: {tamanho_total_gb:.2f} GB')
print(f'🧾 Arquivo de controle: {CONTROLE_ARQUIVOS_CSV}')

if ausentes:
    print('\\n📌 Competências ausentes:')
    print(ausentes)
else:
    print('\\n🎉 Todas as 163 competências foram localizadas.')


✅ CSVs presentes: 163 de 163
❌ CSVs ausentes: 0
💾 Volume aproximado: 0.49 GB
🧾 Arquivo de controle: /content/drive/MyDrive/Suprimentos de Fundos - CPGF/logs/controle_arquivos_cpgf.csv
\n🎉 Todas as 163 competências foram localizadas.


## 1️⃣1️⃣ Verificar os cabeçalhos antes da consolidação

Antes de unir os dados, esta célula percorre os 163 arquivos **sem carregar o conteúdo completo** e compara os cabeçalhos.

A saída mostra:

- codificação detectada;
- separador;
- quantidade de colunas;
- diferenças em relação à amostra de julho/2026;
- quantidade de esquemas distintos encontrada no período.

O relatório completo é salvo em:

`logs/relatorio_esquemas_cpgf.csv`

Se houver alguma diferença histórica de estrutura, ela será registrada. A consolidação posterior fará a união flexível pelo **nome da coluna**.


In [5]:
def analisar_esquemas_periodo() -> List[Dict[str, object]]:
    """Analisa os cabeçalhos de todos os CSVs disponíveis."""
    resultados = []

    for comp in COMPETENCIAS_ESPERADAS:
        caminho = caminho_csv_esperado(comp)

        if not caminho.exists() or caminho.stat().st_size == 0:
            continue

        try:
            info = validar_estrutura_csv(caminho)

            resultados.append({
                'competencia': comp,
                'arquivo': caminho.name,
                'encoding': info['encoding'],
                'separador': info['separador'],
                'numero_colunas': info['numero_colunas'],
                'igual_referencia_conjunto': info['igual_referencia_conjunto'],
                'igual_referencia_ordem': info['igual_referencia_ordem'],
                'colunas_ausentes': ' | '.join(info['colunas_ausentes']),
                'colunas_extras': ' | '.join(info['colunas_extras']),
                'assinatura_esquema': ' || '.join(info['cabecalho']),
                'cabecalho': info['cabecalho'],
            })

        except Exception as e:
            log(f'❌ Erro ao ler cabeçalho de {caminho.name}: {repr(e)}')

    # Relatório tabular sem a lista Python do cabeçalho.
    with open(
        RELATORIO_ESQUEMAS_CSV,
        'w',
        encoding='utf-8-sig',
        newline=''
    ) as f:

        campos = [
            'competencia',
            'arquivo',
            'encoding',
            'separador',
            'numero_colunas',
            'igual_referencia_conjunto',
            'igual_referencia_ordem',
            'colunas_ausentes',
            'colunas_extras',
            'assinatura_esquema',
        ]

        escritor = csv.DictWriter(
            f,
            fieldnames=campos,
            delimiter=';'
        )

        escritor.writeheader()

        for item in resultados:
            escritor.writerow({
                chave: item[chave]
                for chave in campos
            })

    return resultados


resultados_esquemas = analisar_esquemas_periodo()

assinaturas = {}

for item in resultados_esquemas:
    assinatura = item['assinatura_esquema']
    assinaturas.setdefault(assinatura, []).append(item['competencia'])

print(f'✅ Arquivos analisados: {len(resultados_esquemas)}')
print(f'🧩 Esquemas distintos encontrados: {len(assinaturas)}')
print(f'🧾 Relatório: {RELATORIO_ESQUEMAS_CSV}')

for i, (assinatura, competencias) in enumerate(
    assinaturas.items(),
    start=1
):
    print('\\n' + '-' * 80)
    print(f'🧩 Esquema {i}')
    print(f'📆 Competências: {len(competencias)}')
    print(f'🔎 Primeiras competências: {competencias[:10]}')

    colunas = assinatura.split(' || ')
    print(f'📋 Colunas ({len(colunas)}):')
    for coluna in colunas:
        print(f'  - {coluna}')

divergentes = [
    item for item in resultados_esquemas
    if not item['igual_referencia_conjunto']
]

if divergentes:
    print(
        f'\\n⚠️ {len(divergentes)} arquivo(s) possuem conjunto de colunas '
        'diferente da referência 202607.'
    )
else:
    print(
        '\\n🎉 Todos os arquivos analisados possuem o mesmo conjunto de '
        'colunas da referência 202607.'
    )


✅ Arquivos analisados: 163
🧩 Esquemas distintos encontrados: 1
🧾 Relatório: /content/drive/MyDrive/Suprimentos de Fundos - CPGF/logs/relatorio_esquemas_cpgf.csv
\n--------------------------------------------------------------------------------
🧩 Esquema 1
📆 Competências: 163
🔎 Primeiras competências: ['201301', '201302', '201303', '201304', '201305', '201306', '201307', '201308', '201309', '201310']
📋 Colunas (15):
  - CÓDIGO ÓRGÃO SUPERIOR
  - NOME ÓRGÃO SUPERIOR
  - CÓDIGO ÓRGÃO
  - NOME ÓRGÃO
  - CÓDIGO UNIDADE GESTORA
  - NOME UNIDADE GESTORA
  - ANO EXTRATO
  - MÊS EXTRATO
  - CPF PORTADOR
  - NOME PORTADOR
  - CNPJ OU CPF FAVORECIDO
  - NOME FAVORECIDO
  - TRANSAÇÃO
  - DATA TRANSAÇÃO
  - VALOR TRANSAÇÃO
\n🎉 Todos os arquivos analisados possuem o mesmo conjunto de colunas da referência 202607.


## 1️⃣2️⃣ Consolidar todos os CSVs em um único arquivo

Esta etapa cria:

`dados_consolidado/CPGF_201301_a_202607.csv`

### 🧩 Estratégia adotada

A consolidação é feita **incrementalmente**, arquivo a arquivo e em blocos de linhas. Isso evita carregar toda a série histórica na memória do Colab.

Os dados mensais são preservados como texto durante a leitura para evitar conversões indesejadas de códigos, CPFs, CNPJs ou valores monetários.

Duas colunas técnicas são acrescentadas **somente ao arquivo consolidado**:

- `COMPETENCIA_ARQUIVO`: período `AAAAMM` correspondente ao arquivo de origem;
- `ARQUIVO_ORIGEM`: nome do CSV mensal de onde a linha foi lida.

> `COMPETENCIA_ARQUIVO` representa o período do arquivo/extrato e não deve ser confundida com `DATA TRANSAÇÃO`.

### 🔄 Diferenças de estrutura

Antes de iniciar a escrita, o notebook identifica a união de todas as colunas existentes. Caso algum arquivo histórico tenha uma coluna extra ou ausente:

- nenhuma coluna é descartada;
- as colunas são alinhadas pelo nome;
- campos inexistentes naquele mês são preenchidos com vazio.

O arquivo final é gravado em **UTF-8-SIG**, separado por `;`.


In [6]:
# ============================================================
# 🧩 CONSOLIDAÇÃO DOS ARQUIVOS MENSAIS DO CPGF
# ============================================================

def obter_colunas_unificadas(
    caminhos_csv: List[Path]
) -> List[str]:
    """
    Cria a ordem final das colunas.

    Começa pela estrutura de referência de 202607 e acrescenta,
    na ordem em que aparecem, quaisquer colunas históricas extras.
    """

    colunas_finais = list(COLUNAS_REFERENCIA_202607)

    for caminho in caminhos_csv:
        cabecalho, _, _ = ler_cabecalho_csv(caminho)

        for coluna in cabecalho:
            if coluna not in colunas_finais:
                colunas_finais.append(coluna)

    # Colunas técnicas ficam sempre no final.
    for tecnica in [
        'COMPETENCIA_ARQUIVO',
        'ARQUIVO_ORIGEM'
    ]:
        if tecnica not in colunas_finais:
            colunas_finais.append(tecnica)

    return colunas_finais


def consolidar_cpgf(
    competencia_inicial: str = COMPETENCIA_INICIAL,
    competencia_final: str = COMPETENCIA_FINAL,
    chunksize: int = CHUNKSIZE_CONSOLIDACAO,
    exigir_todos: bool = EXIGIR_TODAS_COMPETENCIAS_PARA_CONSOLIDAR
) -> Tuple[Path, pd.DataFrame]:
    """
    Consolida os CSVs mensais de forma incremental.

    O método:
    - preserva os CSVs mensais;
    - não remove duplicidades;
    - lê todas as colunas como string;
    - faz união flexível por nome de coluna;
    - acrescenta colunas técnicas de rastreabilidade;
    - grava UTF-8-SIG com separador ';';
    - utiliza quebra real de linha entre os registros;
    - registra quantidade de linhas por arquivo.
    """

    # ========================================================
    # 📆 Lista das competências
    # ========================================================

    competencias = gerar_competencias(
        competencia_inicial,
        competencia_final
    )

    caminhos = []
    ausentes = []

    for comp in competencias:

        caminho = caminho_csv_esperado(comp)

        if caminho.exists() and caminho.stat().st_size > 0:
            caminhos.append((comp, caminho))
        else:
            ausentes.append(comp)

    # ========================================================
    # ⚠️ Verificação de competências ausentes
    # ========================================================

    if ausentes and exigir_todos:
        raise FileNotFoundError(
            'A consolidação foi interrompida porque ainda há '
            f'{len(ausentes)} competência(s) ausente(s): {ausentes}'
        )

    if not caminhos:
        raise FileNotFoundError(
            'Nenhum CSV mensal foi localizado para consolidação.'
        )

    # ========================================================
    # 🧩 Estrutura final das colunas
    # ========================================================

    colunas_finais = obter_colunas_unificadas(
        [caminho for _, caminho in caminhos]
    )

    # ========================================================
    # 📁 Caminho do arquivo consolidado
    # ========================================================

    saida = (
        CONSOLIDADO_DIR
        / f'CPGF_{competencia_inicial}_a_{competencia_final}.csv'
    )

    # Remove uma versão anterior, caso exista.
    if saida.exists():
        saida.unlink()
        log(
            f'🗑️ Consolidado anterior removido: '
            f'{saida.name}'
        )

    # ========================================================
    # 📊 Variáveis de controle
    # ========================================================

    relatorio = []

    total_linhas = 0

    escreveu_cabecalho = False

    log('🧩 Iniciando consolidação incremental.')
    log(
        f'📄 Arquivos mensais considerados: '
        f'{len(caminhos)}'
    )
    log(
        f'📋 Colunas finais: '
        f'{len(colunas_finais)}'
    )
    log(
        f'💾 Destino: '
        f'{saida}'
    )

    # ========================================================
    # 💾 Escrita incremental do consolidado
    # ========================================================

    # O arquivo permanece aberto durante toda a consolidação,
    # evitando múltiplas marcas BOM e reduzindo operações
    # repetidas de abertura e fechamento.
    with open(
        saida,
        'w',
        encoding='utf-8-sig',
        newline=''
    ) as arquivo_saida:

        for i, (comp, caminho) in enumerate(
            caminhos,
            start=1
        ):

            # ------------------------------------------------
            # 🔎 Identifica estrutura do arquivo mensal
            # ------------------------------------------------

            cabecalho, encoding, separador = (
                ler_cabecalho_csv(caminho)
            )

            linhas_arquivo = 0

            inicio = time.time()

            log(
                f'📌 [{i}/{len(caminhos)}] '
                f'Consolidando {caminho.name} | '
                f'encoding={encoding} | '
                f'sep={repr(separador)}'
            )

            # ------------------------------------------------
            # 📖 Leitura incremental
            # ------------------------------------------------

            leitor_chunks = pd.read_csv(
                caminho,
                sep=separador,
                encoding=encoding,
                dtype=str,
                keep_default_na=False,
                na_filter=False,
                chunksize=chunksize,
                on_bad_lines='error',
            )

            for chunk in leitor_chunks:

                # --------------------------------------------
                # 🧹 Remove eventual BOM residual
                # --------------------------------------------

                if len(chunk.columns) > 0:

                    primeiro = chunk.columns[0]

                    if primeiro.startswith('\ufeff'):
                        chunk = chunk.rename(
                            columns={
                                primeiro:
                                primeiro.lstrip('\ufeff')
                            }
                        )

                # --------------------------------------------
                # 🏷️ Colunas técnicas de rastreabilidade
                # --------------------------------------------

                chunk['COMPETENCIA_ARQUIVO'] = comp

                chunk['ARQUIVO_ORIGEM'] = (
                    caminho.name
                )

                # --------------------------------------------
                # 🧩 União flexível por nome de coluna
                # --------------------------------------------

                chunk = chunk.reindex(
                    columns=colunas_finais,
                    fill_value=''
                )

                # --------------------------------------------
                # 💾 Escrita no CSV consolidado
                # --------------------------------------------

                chunk.to_csv(
                    arquivo_saida,
                    sep=';',
                    index=False,
                    header=not escreveu_cabecalho,

                    # ✅ CORREÇÃO:
                    # quebra REAL de linha entre registros
                    lineterminator='\n',

                    quoting=csv.QUOTE_MINIMAL
                )

                escreveu_cabecalho = True

                # --------------------------------------------
                # 📊 Contagem das linhas
                # --------------------------------------------

                n = len(chunk)

                linhas_arquivo += n
                total_linhas += n

            # ------------------------------------------------
            # ⏱️ Estatísticas da competência
            # ------------------------------------------------

            duracao = time.time() - inicio

            relatorio.append({
                'competencia': comp,
                'arquivo': caminho.name,

                'tamanho_mb': round(
                    caminho.stat().st_size
                    / (1024 ** 2),
                    2
                ),

                'encoding_detectado': encoding,

                'separador_detectado':
                    repr(separador),

                'numero_colunas_origem':
                    len(cabecalho),

                'numero_linhas':
                    linhas_arquivo,

                'duracao_segundos':
                    round(duracao, 2),

                'status':
                    'consolidado',
            })

            log(
                f'✅ {comp}: '
                f'{linhas_arquivo:,} linha(s) '
                f'em {duracao:.1f}s.'
            )

    # ========================================================
    # 🧾 Relatório da consolidação
    # ========================================================

    df_relatorio = pd.DataFrame(relatorio)

    df_relatorio.to_csv(
        RELATORIO_CONSOLIDACAO_CSV,
        sep=';',
        index=False,
        encoding='utf-8-sig',

        # Também deixa explícita a quebra de linha correta.
        lineterminator='\n'
    )

    # ========================================================
    # 📦 Estatísticas finais
    # ========================================================

    tamanho_final_gb = (
        saida.stat().st_size
        / (1024 ** 3)
    )

    log('🎉 Consolidação concluída.')

    log(
        f'📊 Total de linhas de dados: '
        f'{total_linhas:,}'
    )

    log(
        f'💾 Tamanho do consolidado: '
        f'{tamanho_final_gb:.2f} GB'
    )

    log(
        f'🧾 Relatório: '
        f'{RELATORIO_CONSOLIDACAO_CSV}'
    )

    return saida, df_relatorio


# ============================================================
# ▶️ EXECUTAR A CONSOLIDAÇÃO
# ============================================================

caminho_consolidado, relatorio_consolidacao = (
    consolidar_cpgf()
)


# ============================================================
# 📄 RESULTADO
# ============================================================

print('\n' + '=' * 80)
print('📄 ARQUIVO CONSOLIDADO')
print('=' * 80)

print(caminho_consolidado)


print('\n' + '=' * 80)
print('📊 RESUMO DAS PRIMEIRAS COMPETÊNCIAS')
print('=' * 80)

display(
    relatorio_consolidacao.head()
)


# ============================================================
# 🔎 TESTE RÁPIDO DAS QUEBRAS DE LINHA
# ============================================================

print('\n' + '=' * 80)
print('🔎 TESTE DAS PRIMEIRAS LINHAS FÍSICAS DO CSV')
print('=' * 80)

with open(
    caminho_consolidado,
    'r',
    encoding='utf-8-sig'
) as f:

    for numero_linha in range(1, 4):

        linha = f.readline()

        print(
            f'\nLinha {numero_linha}:'
        )

        print(
            repr(linha[:300])
        )


print(
    '\n✅ Se foram exibidas três linhas diferentes, '
    'o arquivo possui as quebras de linha corretamente.'
)

[2026-08-09 23:13:38] 🧩 Iniciando consolidação incremental.
[2026-08-09 23:13:38] 📄 Arquivos mensais considerados: 163
[2026-08-09 23:13:38] 📋 Colunas finais: 17
[2026-08-09 23:13:38] 💾 Destino: /content/drive/MyDrive/Suprimentos de Fundos - CPGF/dados_consolidado/CPGF_201301_a_202607.csv
[2026-08-09 23:13:38] 📌 [1/163] Consolidando 201301_CPGF.csv | encoding=cp1252 | sep=';'
[2026-08-09 23:13:38] ✅ 201301: 17,550 linha(s) em 0.2s.
[2026-08-09 23:13:38] 📌 [2/163] Consolidando 201302_CPGF.csv | encoding=cp1252 | sep=';'
[2026-08-09 23:13:38] ✅ 201302: 3,310 linha(s) em 0.0s.
[2026-08-09 23:13:38] 📌 [3/163] Consolidando 201303_CPGF.csv | encoding=cp1252 | sep=';'
[2026-08-09 23:13:39] ✅ 201303: 9,473 linha(s) em 0.1s.
[2026-08-09 23:13:39] 📌 [4/163] Consolidando 201304_CPGF.csv | encoding=cp1252 | sep=';'
[2026-08-09 23:13:39] ✅ 201304: 16,190 linha(s) em 0.2s.
[2026-08-09 23:13:39] 📌 [5/163] Consolidando 201305_CPGF.csv | encoding=cp1252 | sep=';'
[2026-08-09 23:13:39] ✅ 201305: 19,377 

,competencia,arquivo,tamanho_mb,encoding_detectado,separador_detectado,numero_colunas_origem,numero_linhas,duracao_segundos,status
0,201301,201301_CPGF.csv,5.03,cp1252,';',15,17550,0.22,consolidado
1,201302,201302_CPGF.csv,0.91,cp1252,';',15,3310,0.04,consolidado
2,201303,201303_CPGF.csv,2.64,cp1252,';',15,9473,0.11,consolidado
3,201304,201304_CPGF.csv,4.52,cp1252,';',15,16190,0.19,consolidado
4,201305,201305_CPGF.csv,5.39,cp1252,';',15,19377,0.25,consolidado



🔎 TESTE DAS PRIMEIRAS LINHAS FÍSICAS DO CSV

Linha 1:
'CÓDIGO ÓRGÃO SUPERIOR;NOME ÓRGÃO SUPERIOR;CÓDIGO ÓRGÃO;NOME ÓRGÃO;CÓDIGO UNIDADE GESTORA;NOME UNIDADE GESTORA;ANO EXTRATO;MÊS EXTRATO;CPF PORTADOR;NOME PORTADOR;CNPJ OU CPF FAVORECIDO;NOME FAVORECIDO;TRANSAÇÃO;DATA TRANSAÇÃO;VALOR TRANSAÇÃO;COMPETENCIA_ARQUIVO;ARQUIVO_ORIGEM\n'

Linha 2:
'63000;Advocacia-Geral da União;63000;Advocacia-Geral da União - Unidades com vínculo direto;110161;SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO;2013;01;***.341.701-**;RAIMUNDO RODRIGUES DE OLIVEIRA;-2;NAO SE APLICA;SAQUE CASH/ATM BB;26/11/2012;100,00;201301;201301_CPGF.csv\n'

Linha 3:
'63000;Advocacia-Geral da União;63000;Advocacia-Geral da União - Unidades com vínculo direto;110161;SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO;2013;01;***.402.072-**;JAIME BATISTA DE SOUSA;15282700000193;SELCOM MATERIAIS ELETRICOS LTDA;COMPRA A/V - R$ - APRES;04/12/2012;249,48;201301;201301_CPGF.cs'

✅ Se foram exibidas três linhas diferentes, o arquivo possui as

## 1️⃣3️⃣ Validação final do conjunto consolidado

A última célula produz uma checagem de integridade operacional do processo.

Ela verifica:

1. se as **163 competências** estão presentes;
2. quantidade de linhas de cada CSV mensal, conforme registrada durante a consolidação;
3. tamanho de cada arquivo;
4. estrutura dos cabeçalhos;
5. quantidade total de linhas mensais;
6. quantidade de linhas lidas no arquivo consolidado;
7. quantidade de competências distintas presentes na coluna técnica;
8. existência das colunas técnicas;
9. correspondência entre o total mensal e o total consolidado.

A leitura do consolidado é feita em blocos e somente nas colunas necessárias à validação, reduzindo o uso de memória.

O resultado é salvo em:

`logs/validacao_final_cpgf.csv`


In [12]:
def validar_consolidado(
    caminho_consolidado: Path,
    relatorio_consolidacao: pd.DataFrame
) -> pd.DataFrame:
    """Executa as validações finais do projeto."""
    caminho_consolidado = Path(caminho_consolidado)

    if not caminho_consolidado.exists():
        raise FileNotFoundError(
            f'Consolidado não encontrado: {caminho_consolidado}'
        )

    # ------------------------------------------------------
    # 1. Competências mensais
    # ------------------------------------------------------
    ausentes = listar_competencias_ausentes(
        COMPETENCIA_INICIAL,
        COMPETENCIA_FINAL
    )

    total_arquivos_presentes = (
        len(COMPETENCIAS_ESPERADAS) - len(ausentes)
    )

    # ------------------------------------------------------
    # 2. Total de linhas mensais registrado na consolidação
    # ------------------------------------------------------
    total_linhas_mensais = int(
        relatorio_consolidacao['numero_linhas'].sum()
    )

    # ------------------------------------------------------
    # 3. Cabeçalho do consolidado
    # ------------------------------------------------------
    cabecalho_consolidado, encoding_consolidado, sep_consolidado = (
        ler_cabecalho_csv(caminho_consolidado)
    )

    colunas_tecnicas_ok = all(
        c in cabecalho_consolidado
        for c in ['COMPETENCIA_ARQUIVO', 'ARQUIVO_ORIGEM']
    )

    # ------------------------------------------------------
    # 4. Contagem independente do consolidado
    # ------------------------------------------------------
    total_linhas_consolidado = 0
    competencias_no_consolidado = set()
    arquivos_origem_no_consolidado = set()

    leitor = pd.read_csv(
        caminho_consolidado,
        sep=';',
        encoding='utf-8-sig',
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        usecols=['COMPETENCIA_ARQUIVO', 'ARQUIVO_ORIGEM'],
        chunksize=500_000,
        on_bad_lines='error',
    )

    for chunk in leitor:
        total_linhas_consolidado += len(chunk)

        competencias_no_consolidado.update(
            chunk['COMPETENCIA_ARQUIVO'].unique().tolist()
        )

        arquivos_origem_no_consolidado.update(
            chunk['ARQUIVO_ORIGEM'].unique().tolist()
        )

    totais_batem = (
        total_linhas_consolidado == total_linhas_mensais
    )

    competencias_esperadas_set = set(COMPETENCIAS_ESPERADAS)

    competencias_consolidado_corretas = (
        competencias_no_consolidado == competencias_esperadas_set
        if not ausentes
        else competencias_no_consolidado.issubset(competencias_esperadas_set)
    )

    # ------------------------------------------------------
    # 5. Resumo
    # ------------------------------------------------------
    resumo = pd.DataFrame([
        {
            'indicador': 'competencias_esperadas',
            'valor': len(COMPETENCIAS_ESPERADAS),
            'status': 'OK',
        },
        {
            'indicador': 'arquivos_mensais_presentes',
            'valor': total_arquivos_presentes,
            'status': (
                'OK'
                if total_arquivos_presentes == len(COMPETENCIAS_ESPERADAS)
                else 'ATENÇÃO'
            ),
        },
        {
            'indicador': 'competencias_ausentes',
            'valor': len(ausentes),
            'status': 'OK' if not ausentes else 'ATENÇÃO',
        },
        {
            'indicador': 'linhas_somadas_nos_mensais',
            'valor': total_linhas_mensais,
            'status': 'OK',
        },
        {
            'indicador': 'linhas_lidas_no_consolidado',
            'valor': total_linhas_consolidado,
            'status': 'OK' if totais_batem else 'ERRO',
        },
        {
            'indicador': 'totais_de_linhas_coincidem',
            'valor': totais_batem,
            'status': 'OK' if totais_batem else 'ERRO',
        },
        {
            'indicador': 'competencias_distintas_no_consolidado',
            'valor': len(competencias_no_consolidado),
            'status': (
                'OK'
                if competencias_consolidado_corretas
                else 'ERRO'
            ),
        },
        {
            'indicador': 'arquivos_origem_distintos_no_consolidado',
            'valor': len(arquivos_origem_no_consolidado),
            'status': (
                'OK'
                if len(arquivos_origem_no_consolidado)
                == total_arquivos_presentes
                else 'ATENÇÃO'
            ),
        },
        {
            'indicador': 'colunas_tecnicas_presentes',
            'valor': colunas_tecnicas_ok,
            'status': 'OK' if colunas_tecnicas_ok else 'ERRO',
        },
        {
            'indicador': 'encoding_consolidado_detectado',
            'valor': encoding_consolidado,
            'status': 'OK',
        },
        {
            'indicador': 'separador_consolidado_detectado',
            'valor': repr(sep_consolidado),
            'status': 'OK' if sep_consolidado == ';' else 'ATENÇÃO',
        },
        {
            'indicador': 'tamanho_consolidado_gb',
            'valor': round(
                caminho_consolidado.stat().st_size / (1024 ** 3),
                3
            ),
            'status': 'OK',
        },
    ])

    resumo.to_csv(
        VALIDACAO_FINAL_CSV,
        sep=';',
        index=False,
        encoding='utf-8-sig'
    )

    print('=' * 80)
    print('✅ VALIDAÇÃO FINAL — CPGF')
    print('=' * 80)

    for _, linha in resumo.iterrows():
        print(
            f"{linha['status']:>7} | "
            f"{linha['indicador']}: {linha['valor']}"
        )

    if ausentes:
        print('\\n⚠️ Competências ausentes:')
        print(ausentes)

    print(f'\\n🧾 Validação salva em: {VALIDACAO_FINAL_CSV}')

    return resumo


resumo_validacao = validar_consolidado(
    caminho_consolidado,
    relatorio_consolidacao
)

display(resumo_validacao)

print('\\n📊 Quantidade de linhas por arquivo mensal:')
display(
    relatorio_consolidacao[
        [
            'competencia',
            'arquivo',
            'tamanho_mb',
            'numero_linhas',
            'numero_colunas_origem',
            'encoding_detectado',
            'separador_detectado',
        ]
    ]
)


✅ VALIDAÇÃO FINAL — CPGF
     OK | competencias_esperadas: 163
     OK | arquivos_mensais_presentes: 163
     OK | competencias_ausentes: 0
     OK | linhas_somadas_nos_mensais: 1876087
     OK | linhas_lidas_no_consolidado: 1876087
     OK | totais_de_linhas_coincidem: True
     OK | competencias_distintas_no_consolidado: 163
     OK | arquivos_origem_distintos_no_consolidado: 163
     OK | colunas_tecnicas_presentes: True
     OK | encoding_consolidado_detectado: utf-8-sig
     OK | separador_consolidado_detectado: ';'
     OK | tamanho_consolidado_gb: 0.487
\n🧾 Validação salva em: /content/drive/MyDrive/Suprimentos de Fundos - CPGF/logs/validacao_final_cpgf.csv


,indicador,valor,status
0,competencias_esperadas,163,OK
1,arquivos_mensais_presentes,163,OK
2,competencias_ausentes,0,OK
3,linhas_somadas_nos_mensais,1876087,OK
4,linhas_lidas_no_consolidado,1876087,OK
5,totais_de_linhas_coincidem,True,OK
6,competencias_distintas_no_consolidado,163,OK
7,arquivos_origem_distintos_no_consolidado,163,OK
8,colunas_tecnicas_presentes,True,OK
9,encoding_consolidado_detectado,utf-8-sig,OK


\n📊 Quantidade de linhas por arquivo mensal:


,competencia,arquivo,tamanho_mb,numero_linhas,numero_colunas_origem,encoding_detectado,separador_detectado
0,201301,201301_CPGF.csv,5.03,17550,15,cp1252,';'
1,201302,201302_CPGF.csv,0.91,3310,15,cp1252,';'
2,201303,201303_CPGF.csv,2.64,9473,15,cp1252,';'
3,201304,201304_CPGF.csv,4.52,16190,15,cp1252,';'
4,201305,201305_CPGF.csv,5.39,19377,15,cp1252,';'
...,...,...,...,...,...,...,...
158,202603,202603_CPGF.csv,2.05,7729,15,cp1252,';'
159,202604,202604_CPGF.csv,3.89,14741,15,cp1252,';'
160,202605,202605_CPGF.csv,3.51,13437,15,cp1252,';'
161,202606,202606_CPGF.csv,4.95,18793,15,cp1252,';'


## ✅ Resultado esperado

Ao final do notebook, a pasta principal deverá conter os arquivos:

`201301_CPGF.csv`  
`201302_CPGF.csv`  
`...`  
`202607_CPGF.csv`

A subpasta `dados_consolidado` deverá conter:

`CPGF_201301_a_202607.csv`

e a pasta `logs` registrará os controles de:

- downloads;
- competências presentes e ausentes;
- diferenças de estrutura;
- número de linhas por arquivo;
- validação final.

### 🔎 Observação para as próximas etapas da pesquisa

Os arquivos mensais permanecem como **camada bruta e reproduzível**. Recomenda-se que qualquer limpeza, transformação, criação de variáveis, aplicação da Lei de Newcomb-Benford ou execução das trilhas de análise seja realizada posteriormente em outro notebook, preservando esta etapa exclusivamente como pipeline de aquisição e consolidação dos dados públicos.
